In [3]:
import os

print(os.listdir())

['.cache', '.codex', '.conda', '.condarc', '.config', '.ipynb_checkpoints', '.ipython', '.jupyter', '.keras', '.matplotlib', '.ms-ad', '.vscode', '.vscode-shared', '1 Multiple Linear Regression.ipynb', '1 Random Forest.ipynb', '1 RF Deployment-Phase=2 - Jupyter Notebook.htm', '1 RF Deployment-Phase=2.ipynb', '1 SLR.ipynb', '1.Assignment GridSearchCV RF.ipynb', '1.Decision_Tree.ipynb', '1.SVMR.ipynb', '1.SVR-Grid.ipynb', '2 Deployment-Phase2.1.ipynb', '2 deployment-Phase2.ipynb', '2.Decision Tree.ipynb', '2.Multiple Linear Regression - Jupyter Notebook.htm', '2.Multiple Linear Regression.ipynb', '2.Random Forest Regression(1).ipynb', '2.Random Forest Regression.ipynb', '3 Deployment-Phase2.ipynb', '3.SVM.ipynb', '3D Objects', '50_Startups.csv', 'activate', 'AdaBoostRegressor.ipynb', 'Anaconda3', 'AppData', 'Application Data', 'Assignment GridSearchCV DT.ipynb', 'Assignment-GridSearchCV SVM.ipynb', 'ChronicKidneyDisease (1).csv', 'ChronicKidneyDisease.csv', 'ckd incomplete.ipynb', 'ckdpr

In [4]:
import pandas as pd

df = pd.read_csv('KARURVYSYA.NS_stock_data.csv')

print(df.head())
print(df.columns)

   Unnamed: 0      open      high       low     close  adjclose    volume  \
0  1996-01-01  2.367385  2.362865  2.360354  2.362865  1.383404   44802.0   
1  1996-01-02  2.362865  2.380442  2.287534  2.290045  1.340769  129429.0   
2  1996-01-03  2.290045  2.350310  2.310133  2.310133  1.352531   94583.0   
3  1996-01-04  2.259913  2.300089  2.249869  2.249869  1.317247   49780.0   
4  1996-01-05  2.249869  2.219737  2.209693  2.210195  1.294019   49780.0   

          ticker  
0  KARURVYSYA.NS  
1  KARURVYSYA.NS  
2  KARURVYSYA.NS  
3  KARURVYSYA.NS  
4  KARURVYSYA.NS  
Index(['Unnamed: 0', 'open', 'high', 'low', 'close', 'adjclose', 'volume',
       'ticker'],
      dtype='object')


In [5]:
# Date column-ஐ datetime format-க்கு மாற்றுதல்
df['Date'] = pd.to_datetime(df['Unnamed: 0'])

# Date அடிப்படையில் data-வை வரிசைப்படுத்துதல்
df = df.sort_values('Date')

# Close price series
price_series = df.set_index('Date')['close']

# முதல் 5 மற்றும் கடைசி 5 values பார்க்க
print(price_series.head())
print(price_series.tail())

Date
1996-01-01    2.362865
1996-01-02    2.290045
1996-01-03    2.310133
1996-01-04    2.249869
1996-01-05    2.210195
Name: close, dtype: float64
Date
2024-04-24    189.600006
2024-04-25    191.250000
2024-04-26    191.199997
2024-04-29    196.199997
2024-04-30    203.949997
Name: close, dtype: float64


In [6]:
from statsmodels.tsa.arima.model import ARIMA

# ARIMA(1,1,1)
model_111 = ARIMA(price_series, order=(1,1,1))
fit_111 = model_111.fit()

# ARIMA(0,1,1)
model_011 = ARIMA(price_series, order=(0,1,1))
fit_011 = model_011.fit()

# AIC மற்றும் BIC
print("ARIMA(1,1,1)")
print("AIC :", fit_111.aic)
print("BIC :", fit_111.bic)

print("\nARIMA(0,1,1)")
print("AIC :", fit_011.aic)

C:\Users\GOD\Anaconda3\envs\timeseries\lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, freq)
C:\Users\GOD\Anaconda3\envs\timeseries\lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, freq)
C:\Users\GOD\Anaconda3\envs\timeseries\lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, freq)
C:\Users\GOD\Anaconda3\envs\timeseries\lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so 

ARIMA(1,1,1)
AIC : 28018.61787054119
BIC : 28039.23154447442

ARIMA(0,1,1)
AIC : 28016.723343233905


In [7]:
print("BIC :", fit_011.bic)

BIC : 28030.465792522726


In [9]:
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error
import numpy as np

# Date index-ஐ சரியாக அமைத்தல்
price_series = price_series.copy()
price_series.index = pd.to_datetime(price_series.index)
price_series = price_series.sort_index()

# 80% Training, 20% Testing
train_size = int(len(price_series) * 0.8)

train = price_series.iloc[:train_size]
test = price_series.iloc[train_size:]
# ARIMA(1,1,1)
# -----------------------------
model_111 = ARIMA(train.values, order=(1,1,1))
fit_111 = model_111.fit()

forecast_111 = fit_111.forecast(steps=len(test))

rmse_111 = np.sqrt(
    mean_squared_error(test.values, forecast_111)
)

# -----------------------------
# ARIMA(0,1,1)
# -----------------------------
model_011 = ARIMA(train.values, order=(0,1,1))
fit_011 = model_011.fit()

forecast_011 = fit_011.forecast(steps=len(test))

rmse_011 = np.sqrt(
    mean_squared_error(test.values, forecast_011)
)

# -----------------------------
# RMSE Results
# -----------------------------
print("ARIMA(1,1,1) RMSE :", rmse_111)
print("ARIMA(0,1,1) RMSE :", rmse_011)

ARIMA(1,1,1) RMSE : 43.11374756072542
ARIMA(0,1,1) RMSE : 43.1032116274746


In [ ]:
உங்கள் Project-ல் எழுதக்கூடிய முடிவு

“KVB share price data-க்கு ARIMA(1,1,1) மற்றும் ARIMA(0,1,1) models ஒப்பிடப்பட்டன. AIC, BIC மற்றும் RMSE ஆகிய மூன்று அளவுகோல்களிலும் ARIMA(0,1,1) குறைந்த மதிப்புகளைப் பெற்றது. எனவே, KVB share price forecasting-க்கு ARIMA(0,1,1) சிறந்த model ஆக தேர்வு செய்யப்படுகிறது.”